# NeoOLAF — OFFLINE micro evaluation for EventStoryLine + FinCausal

This notebook performs **zero LLM/API calls**.

It uses only the already-generated full-run artifacts under:

`examples/RAGTreeDatasets/runs/full_process_isolated_eventstoryline_fincausal_v1`

For both datasets it recomputes from the persisted per-document
`posthoc_evaluation.json` files:

- relation TP / FP / FN
- relation predicted / gold counts
- **micro precision**
- **micro recall**
- **micro F1**
- endpoint TP / FP / FN
- endpoint micro precision / recall / F1
- macro document F1 (diagnostic only)
- positive-gold-document macro F1
- runtime statistics from already-saved `elapsed_seconds`

No NeoOLAF layer is executed and no OpenRouter key is required.


In [1]:
from pathlib import Path
import json, re, statistics
from pprint import pprint

def find_project_root():
    import os
    candidates = []
    env = os.environ.get("NEOOLAF_PROJECT_ROOT")
    if env:
        candidates.append(Path(env))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.append(Path(r"C:\Users\galencarmedeiro\NeoOLAF"))
    for p in candidates:
        if (p / "src" / "neoolaf").is_dir() and (p / "examples").is_dir():
            return p.resolve()
    raise FileNotFoundError(
        "NeoOLAF project root not found. Set NEOOLAF_PROJECT_ROOT."
    )

def safe_dir_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))[:120]

PROJECT_ROOT = find_project_root()
RUNS_ROOT = (
    PROJECT_ROOT
    / "examples"
    / "RAGTreeDatasets"
    / "runs"
    / "full_process_isolated_eventstoryline_fincausal_v1"
)
PROGRESS_PATH = RUNS_ROOT / "full_progress.json"

assert PROGRESS_PATH.is_file(), (
    "Existing full-run progress not found.",
    PROGRESS_PATH,
)

progress = json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUNS_ROOT:", RUNS_ROOT)
print("No API key is read by this notebook.")


PROJECT_ROOT: C:\Users\galencarmedeiro\NeoOLAF
RUNS_ROOT: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\full_process_isolated_eventstoryline_fincausal_v1
No API key is read by this notebook.


## 1. Verify that both full datasets are complete

This prevents accidentally reporting a partial aggregate as a full result.


In [2]:
EXPECTED = {
    "eventstoryline": {
        "total": 443,
        "version": "v1.7",
    },
    "fincausal": {
        "total": 967,
        "version": "unified-v1.3.1-selection-hotfix",
    },
}

for dataset_key, expected in EXPECTED.items():
    ds = progress["datasets"][dataset_key]
    completed = len(ds.get("completed_record_keys") or [])

    print(
        f"{dataset_key:15s}: "
        f"{completed}/{ds['total_records']} "
        f"| version={ds['version']}"
    )

    assert ds["total_records"] == expected["total"], (
        dataset_key,
        ds["total_records"],
        expected["total"],
    )
    assert ds["version"] == expected["version"], (
        dataset_key,
        ds["version"],
        expected["version"],
    )
    assert completed == expected["total"], (
        f"{dataset_key} is not complete; refusing to label its aggregate as full.",
        completed,
        expected["total"],
    )

print("\nBoth datasets are complete.")


eventstoryline : 443/443 | version=v1.7
fincausal      : 967/967 | version=unified-v1.3.1-selection-hotfix

Both datasets are complete.


## 2. Load every persisted post-hoc evaluation

EventStoryLine and FinCausal use slightly different metric key names, so the
loader accepts both schemas (`tp` vs `true_positive`, etc.).


In [3]:
def result_path(dataset_key, record_key):
    return (
        RUNS_ROOT
        / dataset_key
        / "full"
        / safe_dir_name(record_key)
        / "posthoc_evaluation.json"
    )

def metric_count(metrics, *names):
    if not isinstance(metrics, dict):
        return 0
    for name in names:
        if name in metrics and metrics[name] is not None:
            return int(metrics[name] or 0)
    return 0

def normalize_counts(metrics):
    tp = metric_count(metrics, "tp", "true_positive")
    fp = metric_count(metrics, "fp", "false_positive")
    fn = metric_count(metrics, "fn", "false_negative")
    pred = metric_count(metrics, "pred", "predicted")
    gold = metric_count(metrics, "gold", "gold_unique")

    # Defensive reconstruction when a schema omits explicit pred/gold totals.
    if pred == 0 and (tp + fp) > 0:
        pred = tp + fp
    if gold == 0 and (tp + fn) > 0:
        gold = tp + fn

    return {
        "pred": pred,
        "gold": gold,
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }

def load_results(dataset_key):
    ds = progress["datasets"][dataset_key]
    completed_keys = list(ds.get("completed_record_keys") or [])

    results = []
    missing = []

    for rkey in completed_keys:
        p = result_path(dataset_key, rkey)
        if not p.is_file():
            missing.append((rkey, str(p)))
            continue

        row = json.loads(p.read_text(encoding="utf-8"))
        assert row.get("record_key") == rkey, (
            dataset_key,
            rkey,
            row.get("record_key"),
        )
        results.append(row)

    if missing:
        raise RuntimeError(
            f"{dataset_key}: {len(missing)} completed records have no "
            f"posthoc_evaluation.json. First examples: {missing[:5]}"
        )

    assert len(results) == EXPECTED[dataset_key]["total"], (
        dataset_key,
        len(results),
        EXPECTED[dataset_key]["total"],
    )
    return results

results = {
    "eventstoryline": load_results("eventstoryline"),
    "fincausal": load_results("fincausal"),
}

for k, rows in results.items():
    print(k, "loaded evaluations:", len(rows))


eventstoryline loaded evaluations: 443
fincausal loaded evaluations: 967


## 3. Recompute MICRO metrics from global TP / FP / FN

This is the important part.

For micro-F1 we **do not average document F1 values**. We first sum all
TP/FP/FN across the complete dataset and then compute:

- `P = TP / (TP + FP)`
- `R = TP / (TP + FN)`
- `F1 = 2PR / (P + R)`


In [4]:
def prf(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0.0
    )
    return precision, recall, f1

def aggregate(dataset_key, rows):
    relation_counts = [
        normalize_counts(r.get("relation_metrics") or {})
        for r in rows
    ]
    endpoint_counts = [
        normalize_counts(r.get("endpoint_metrics") or {})
        for r in rows
    ]

    def sum_counts(items):
        return {
            name: sum(x[name] for x in items)
            for name in ["pred", "gold", "tp", "fp", "fn"]
        }

    relation = sum_counts(relation_counts)
    endpoint = sum_counts(endpoint_counts)

    rp, rr, rf1 = prf(
        relation["tp"],
        relation["fp"],
        relation["fn"],
    )
    ep, er, ef1 = prf(
        endpoint["tp"],
        endpoint["fp"],
        endpoint["fn"],
    )

    relation.update({
        "precision": rp,
        "recall": rr,
        "micro_f1": rf1,
    })
    endpoint.update({
        "precision": ep,
        "recall": er,
        "micro_f1": ef1,
    })

    # Diagnostic macro values only; NOT the main benchmark number.
    doc_f1 = [
        float((r.get("relation_metrics") or {}).get("f1", 0.0) or 0.0)
        for r in rows
    ]
    positive_doc_f1 = [
        float((r.get("relation_metrics") or {}).get("f1", 0.0) or 0.0)
        for r, c in zip(rows, relation_counts)
        if c["gold"] > 0
    ]

    elapsed = [
        float(r["elapsed_seconds"])
        for r in rows
        if r.get("elapsed_seconds") is not None
    ]

    return {
        "dataset": dataset_key,
        "documents": len(rows),
        "version": EXPECTED[dataset_key]["version"],
        "relation": relation,
        "endpoint": endpoint,
        "macro_doc_f1": (
            sum(doc_f1) / len(doc_f1)
            if doc_f1 else None
        ),
        "macro_positive_gold_doc_f1": (
            sum(positive_doc_f1) / len(positive_doc_f1)
            if positive_doc_f1 else None
        ),
        "runtime": {
            "documents_with_elapsed_seconds": len(elapsed),
            "mean_seconds": statistics.mean(elapsed) if elapsed else None,
            "median_seconds": statistics.median(elapsed) if elapsed else None,
            "min_seconds": min(elapsed) if elapsed else None,
            "max_seconds": max(elapsed) if elapsed else None,
        },
    }

aggregate_results = {
    k: aggregate(k, rows)
    for k, rows in results.items()
}

for k in ["eventstoryline", "fincausal"]:
    a = aggregate_results[k]
    r = a["relation"]
    e = a["endpoint"]

    print("\n" + "=" * 72)
    print(k.upper(), f"({a['documents']} documents)")
    print("=" * 72)

    print(
        "RELATION MICRO\n"
        f"  predicted = {r['pred']}\n"
        f"  gold      = {r['gold']}\n"
        f"  TP        = {r['tp']}\n"
        f"  FP        = {r['fp']}\n"
        f"  FN        = {r['fn']}\n"
        f"  Precision = {r['precision']:.9f} ({100*r['precision']:.4f}%)\n"
        f"  Recall    = {r['recall']:.9f} ({100*r['recall']:.4f}%)\n"
        f"  MICRO-F1  = {r['micro_f1']:.9f} ({100*r['micro_f1']:.4f}%)"
    )

    print(
        "\nENDPOINT MICRO\n"
        f"  predicted = {e['pred']}\n"
        f"  gold      = {e['gold']}\n"
        f"  TP        = {e['tp']}\n"
        f"  FP        = {e['fp']}\n"
        f"  FN        = {e['fn']}\n"
        f"  Precision = {e['precision']:.9f}\n"
        f"  Recall    = {e['recall']:.9f}\n"
        f"  MICRO-F1  = {e['micro_f1']:.9f}"
    )

    print(
        "\nDIAGNOSTIC MACRO\n"
        f"  mean document F1          = {a['macro_doc_f1']:.9f}\n"
        f"  positive-gold-doc mean F1 = {a['macro_positive_gold_doc_f1']:.9f}"
    )

    print("\nRUNTIME FROM SAVED RESULTS")
    pprint(a["runtime"])



EVENTSTORYLINE (443 documents)
RELATION MICRO
  predicted = 5993
  gold      = 9640
  TP        = 902
  FP        = 5091
  FN        = 8738
  Precision = 0.150508927 (15.0509%)
  Recall    = 0.093568465 (9.3568%)
  MICRO-F1  = 0.115396917 (11.5397%)

ENDPOINT MICRO
  predicted = 2640
  gold      = 5192
  TP        = 2633
  FP        = 7
  FN        = 2559
  Precision = 0.997348485
  Recall    = 0.507126348
  MICRO-F1  = 0.672369765

DIAGNOSTIC MACRO
  mean document F1          = 0.111308323
  positive-gold-doc mean F1 = 0.111308323

RUNTIME FROM SAVED RESULTS
{'documents_with_elapsed_seconds': 443,
 'max_seconds': 1151.4479868999988,
 'mean_seconds': 164.72288780835237,
 'median_seconds': 123.84735869999713,
 'min_seconds': 1.7767012999975123}

FINCAUSAL (967 documents)
RELATION MICRO
  predicted = 276
  gold      = 929
  TP        = 231
  FP        = 45
  FN        = 698
  Precision = 0.836956522 (83.6957%)
  Recall    = 0.248654467 (24.8654%)
  MICRO-F1  = 0.383402490 (38.3402%)

EN

## 4. Compact comparison table and JSON export

The table keeps **micro-F1** separate from macro document F1 so they cannot be
accidentally mixed in the paper.


In [5]:
import pandas as pd

table_rows = []
for k in ["eventstoryline", "fincausal"]:
    a = aggregate_results[k]
    r = a["relation"]
    e = a["endpoint"]
    rt = a["runtime"]

    table_rows.append({
        "dataset": k,
        "documents": a["documents"],
        "relation_predicted": r["pred"],
        "relation_gold": r["gold"],
        "relation_tp": r["tp"],
        "relation_fp": r["fp"],
        "relation_fn": r["fn"],
        "relation_micro_precision": r["precision"],
        "relation_micro_recall": r["recall"],
        "relation_micro_f1": r["micro_f1"],
        "relation_macro_doc_f1": a["macro_doc_f1"],
        "relation_macro_positive_gold_doc_f1": a["macro_positive_gold_doc_f1"],
        "endpoint_micro_precision": e["precision"],
        "endpoint_micro_recall": e["recall"],
        "endpoint_micro_f1": e["micro_f1"],
        "mean_saved_elapsed_seconds": rt["mean_seconds"],
        "median_saved_elapsed_seconds": rt["median_seconds"],
    })

df = pd.DataFrame(table_rows)
display(df)

OUTPUT_JSON = RUNS_ROOT / "OFFLINE_MICRO_EVAL_ESL_FINCAUSAL.json"
OUTPUT_CSV = RUNS_ROOT / "OFFLINE_MICRO_EVAL_ESL_FINCAUSAL.csv"

OUTPUT_JSON.write_text(
    json.dumps(aggregate_results, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
df.to_csv(OUTPUT_CSV, index=False)

print("\nSaved:")
print(" -", OUTPUT_JSON)
print(" -", OUTPUT_CSV)
print("\nZERO API / ZERO LLM execution completed.")


,dataset,documents,relation_predicted,relation_gold,relation_tp,relation_fp,relation_fn,relation_micro_precision,relation_micro_recall,relation_micro_f1,relation_macro_doc_f1,relation_macro_positive_gold_doc_f1,endpoint_micro_precision,endpoint_micro_recall,endpoint_micro_f1,mean_saved_elapsed_seconds,median_saved_elapsed_seconds
0,eventstoryline,443,5993,9640,902,5091,8738,0.150509,0.093568,0.115397,0.111308,0.111308,0.997348,0.507126,0.67237,164.722888,123.847359
1,fincausal,967,276,929,231,45,698,0.836957,0.248654,0.383402,0.238883,0.248654,1.000000,1.000000,1.00000,20.631212,16.384426



Saved:
 - C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\full_process_isolated_eventstoryline_fincausal_v1\OFFLINE_MICRO_EVAL_ESL_FINCAUSAL.json
 - C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\full_process_isolated_eventstoryline_fincausal_v1\OFFLINE_MICRO_EVAL_ESL_FINCAUSAL.csv

ZERO API / ZERO LLM execution completed.
